In [ ]:
import sys
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
sys.path.append("../../../")
from src.data.data_splits import generate_split_mask
from src.utils import get_label_list
np.random.seed(42)
plt.rcParams["font.size"] = 7

In [ ]:
data_root = Path("../../../data/raw/mimic-iv-ed-benchmark") # follow instructions from https://github.com/nliulab/mimic4ed-benchmark to prepare data from MIMIC-IV
tr_df = pd.read_csv(data_root / "train.csv")
te_df = pd.read_csv(data_root / "test.csv")
data_df = pd.concat((tr_df, te_df))

In [ ]:
data_df

In [ ]:
data_df = data_df.rename(columns={'gender': 'sex'})
data_df.sex.value_counts()

In [ ]:
data_df.age.plot.hist(figsize=(3, 2))

In [ ]:
label_names = get_label_list("mimic-iv-ed")
label_names

In [ ]:
def get_info(df):
    return len(df), df['subject_id'].nunique()

In [ ]:
totalinfo = get_info(data_df)
print(f"Total: {totalinfo[0]} EHRs, {totalinfo[1]} unique patients")

In [ ]:
# patient-wise train-val split
val_test_pids = pd.Series(data_df.subject_id.unique()).sample(n=15_000, random_state=42, replace=False)
val_pids = val_test_pids[:5_000]
test_pids = val_test_pids[5_000:]
traindf = data_df[~data_df.subject_id.isin(val_test_pids)]
val_df = data_df[data_df.subject_id.isin(val_pids)]
test_df = data_df[data_df.subject_id.isin(test_pids)]
traindf.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

In [ ]:
traininfo = get_info(traindf)
print(f"train: {traininfo[0]} records, {traininfo[1]} unique patients")

In [ ]:
# plot #records per patient
import scipy
import matplotlib.pyplot as plt

plt.figure(figsize=(2.5, 1.75))
counts = traindf.groupby("subject_id").size().values
mean_count = np.mean(counts)
median_count = np.median(counts)
percentile = np.percentile(counts, 90)
# empirical survival function of counts
res = scipy.stats.ecdf(counts)
res.sf.plot(color="blue")
plt.axvline(mean_count, color="red", label=f"Mean: {mean_count:.2f}")
plt.axvline(median_count, color="green", label=f"Median: {median_count}")
plt.axvline(percentile, color="orange", label=f"90th perc: {percentile}")
plt.xlabel("Number of Records per Patient")
plt.ylabel("1 - Cumulative Probability")
plt.legend()
plt.yscale("log")
plt.title("MIMIC-IV-ED: #Record per Patient")
# plt.savefig(
#     "../../../figs/mimic-iv-ed/mimic-iv-ed_record_stats.pdf", bbox_inches="tight"
# )

In [ ]:
# temporal patient-specific split
eval_mask = generate_split_mask(
    traindf,
    patient_id_col="subject_id",
    timestamp_col="outtime",
    label_cols=label_names,
    n_holdout_classes=1,
)

train_historical_df = traindf[~eval_mask]
train_future_df = traindf[eval_mask]

In [ ]:
train_historical_info = get_info(train_historical_df)
train_future_info = get_info(train_future_df)
val_info = get_info(val_df)
testinfo = get_info(test_df)
print(f"Train(historical): {train_historical_info[0]} records, {train_historical_info[1]} unique patients")
print(f"Train(future): {train_future_info[0]} records, {train_future_info[1]} unique patients")
print(f"Val: {val_info[0]} records, {val_info[1]} unique patients")
print(f"Test: {testinfo[0]} records, {testinfo[1]} unique patients")

In [ ]:
# we create a helper dataframe that indicates any diseases present per patient in the training set
present_diseases_by_patient = train_historical_df.groupby('subject_id')[label_names].max()
present_diseases_by_patient

In [ ]:
from typing import List

# we generate two new columns for each future record
    # 'seen_diseases': comma-separated list of disease names that were already present in the historical records of the same patient
    # 'unseen_diseases': comma-separated list of disease names that are present in the future record but were not present in the historical records of the same patient

def case_stratification_by_historical_disease_presence(row:pd.Series, historical_diseases_by_patient:pd.DataFrame, label_cols:List[str], patient_id_col:str="subject_id", verbose:bool=False) -> bool:
    pid = row[patient_id_col]
    label = row[label_cols]
    present_diseases_historical = historical_diseases_by_patient.loc[pid]
    assert present_diseases_historical.shape == label.shape, f"Shape mismatch: {present_diseases_historical.shape} vs {label.shape}"
    positive_unseen = label[(label == 1) & (present_diseases_historical == 0)]
    positive_seen = label[((label == 1) & (present_diseases_historical == 1)) | ((label == 0) & (present_diseases_historical == 0))]
    print(f"Patient {pid} has unseen diseases: {positive_unseen.index.tolist()}") if verbose and len(positive_unseen) > 0 else None
    print(f"Patient {pid} has seen diseases: {positive_seen.index.tolist()}") if verbose and len(positive_seen) > 0 else None
    unseen_labels = positive_unseen.index.tolist() if len(positive_unseen) > 0 else []
    seen_labels = positive_seen.index.tolist() if len(positive_seen) > 0 else []
    assert set(unseen_labels).isdisjoint(set(seen_labels)), f"Seen and unseen labels should be disjoint: {unseen_labels} vs {seen_labels}"
    unseen_labels_str = ",".join(unseen_labels)
    seen_labels_str = ",".join(seen_labels)
    return pd.Series([seen_labels_str, unseen_labels_str]) # must return pd.Series to fill two columns simultaneously

In [ ]:
train_future_df[['seen_diseases', 'unseen_diseases']] = train_future_df.progress_apply(
    lambda row: case_stratification_by_historical_disease_presence(
        row,
        historical_diseases_by_patient=present_diseases_by_patient,
        label_cols=label_names,
        patient_id_col="subject_id",
        verbose=False
    ),
    axis=1,
    result_type='expand'
)
train_future_df['has_unseen_disease'] = train_future_df['unseen_diseases'].apply(lambda x: len(x) > 0)
train_future_df['has_seen_disease'] = train_future_df['seen_diseases'].apply(lambda x: len(x) > 0)

In [ ]:
train_future_df.has_unseen_disease.value_counts()

In [ ]:
train_future_df.has_seen_disease.value_counts()

In [ ]:
train_future_df.unseen_diseases.value_counts().head(10)

In [ ]:
train_future_df.seen_diseases.value_counts()

In [ ]:
split_names = ["train", "long_eval", "val", "test"]
for n, d in zip(split_names, [train_historical_df, train_future_df, val_df, test_df]):
    print(f"Dataset Info ({n}):")
    info = get_info(d)
    print(f"{info[0]} records, {info[1]} unique patients")
    for label in label_names:
        print(d[label].value_counts(sort=False))
    print("\n")

In [ ]:
csv_root = Path("../../../data/csv")
train_historical_df.to_csv(csv_root / "mimic-iv-ed_train_historical.csv", index=False)
val_df.to_csv(csv_root / "mimic-iv-ed_val.csv", index=False)
train_future_df.to_csv(csv_root / "mimic-iv-ed_train_future.csv", index=False)
test_df.to_csv(csv_root / "mimic-iv-ed_test.csv", index=False)